In [ ]:
import sys
sys.path.insert(0, ".")          # so Python can find the src/ folder

from src.config import TOP_N, RANDOM_STATE
from src.utils import set_seed, logger

set_seed(RANDOM_STATE)
logger.info(f"Config loaded — TOP_N={TOP_N}")
print("Step 1 complete ✅")

In [ ]:
import sys
sys.path.insert(0, ".")

from src.utils import set_seed
from src.config import RANDOM_STATE
from src.data_loader import load_raw_data, validate_data

set_seed(RANDOM_STATE)

df = load_raw_data()
validate_data(df)

# Show all actual column names so we can confirm
print("All columns:", list(df.columns))

In [ ]:
code = '''# src/preprocessor.py
import re
import pandas as pd
import numpy as np
from src.config import RANDOM_STATE, CLEAN_CSV, DATA_PROC_DIR
from src.utils import logger, ensure_dirs, set_seed

set_seed(RANDOM_STATE)

LEVEL_MAP = {
    "beginner"            : "Beginner",
    "beginner level"      : "Beginner",
    "easy"                : "Beginner",
    "intermediate"        : "Intermediate",
    "intermediate level"  : "Intermediate",
    "medium"              : "Intermediate",
    "advanced"            : "Advanced",
    "advanced level"      : "Advanced",
    "hard"                : "Advanced",
    "expert"              : "Advanced",
    "all"                 : "All Levels",
    "all levels"          : "All Levels",
    "appropriate for all" : "All Levels",
}

def _drop_bad_rows(df):
    before = len(df)
    df = df.dropna(subset=["title"])
    df = df[df["title"].str.strip().ne("")]
    df = df.drop_duplicates(subset=["title", "platform"])
    after = len(df)
    logger.info(f"Dropped {before - after:,} bad/duplicate rows → {after:,} remain")
    return df.reset_index(drop=True)

def _clean_text(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r"\\{|\\}", "", text)
    text = re.sub(r\'\\"\', "", text)
    text = re.sub(r"[^a-zA-Z0-9\\s,\\.\\-\\+\\#]", " ", text)
    text = re.sub(r"\\s+", " ", text).strip()
    return text

def _build_skills(df):
    def combine_row(row):
        parts = []
        for col in ["skills", "associatedskills", "subject", "description"]:
            val = row.get(col, "")
            cleaned = _clean_text(str(val) if pd.notna(val) else "")
            if cleaned:
                parts.append(cleaned)
        return " , ".join(parts)
    df["skills_clean"] = df.apply(combine_row, axis=1)
    logger.info("Built skills_clean")
    return df

def _normalize_level(df):
    def map_level(val):
        if pd.isna(val):
            return "All Levels"
        key = str(val).strip().lower()
        return LEVEL_MAP.get(key, "All Levels")
    df["level_clean"] = df["level"].apply(map_level)
    logger.info(f"Level distribution:\\n{df[\'level_clean\'].value_counts().to_string()}")
    return df

def _normalize_rating(df):
    df["rating"] = pd.to_numeric(df["rating"], errors="coerce")
    global_median = df["rating"].median()
    df["rating_clean"] = df.groupby("platform")["rating"].transform(
        lambda x: x.fillna(x.median() if x.notna().any() else global_median)
    )
    df["rating_clean"] = df["rating_clean"].fillna(global_median).round(2)
    logger.info(f"Rating median: {global_median:.2f}")
    return df

def _normalize_duration(df):
    df["duration_clean"] = df["duration"].fillna("Unknown").astype(str).str.strip()
    return df

def _clean_title(df):
    df["title_clean"] = df["title"].apply(_clean_text)
    return df

def _build_text_features(df):
    df["text_features"] = (
        df["title_clean"].fillna("") + " " +
        df["skills_clean"].fillna("") + " " +
        df["level_clean"].fillna("") + " " +
        df["platform"].fillna("")
    )
    df["text_features"] = df["text_features"].str.replace(r"\\s+", " ", regex=True).str.strip()
    logger.info("Built text_features")
    return df

FINAL_COLS = [
    "title_clean", "platform", "level_clean",
    "rating_clean", "duration_clean",
    "skills_clean", "text_features",
    "provider", "description"
]

def _select_final_columns(df):
    available = [c for c in FINAL_COLS if c in df.columns]
    df = df[available].copy()
    df = df.rename(columns={
        "title_clean"    : "title",
        "level_clean"    : "level",
        "rating_clean"   : "rating",
        "duration_clean" : "duration",
    })
    df = df.reset_index(drop=True)
    df["course_id"] = df.index
    return df

def preprocess(df):
    logger.info("=== Starting Preprocessing Pipeline ===")
    df = _drop_bad_rows(df)
    df = _clean_title(df)
    df = _build_skills(df)
    df = _normalize_level(df)
    df = _normalize_rating(df)
    df = _normalize_duration(df)
    df = _build_text_features(df)
    df = _select_final_columns(df)
    ensure_dirs(DATA_PROC_DIR)
    df.to_csv(CLEAN_CSV, index=False)
    logger.info(f"Saved → {CLEAN_CSV} | Shape: {df.shape}")
    return df

def print_summary(df):
    print("\\n" + "="*55)
    print("       PREPROCESSING SUMMARY")
    print("="*55)
    print(f"  Total clean courses : {len(df):,}")
    print(f"  Columns             : {list(df.columns)}")
    print(f"\\n  Platform breakdown:")
    for p, c in df["platform"].value_counts().items():
        print(f"    {p:<15} → {c:>6,}")
    print(f"\\n  Level breakdown:")
    for l, c in df["level"].value_counts().items():
        print(f"    {l:<20} → {c:>6,}")
    print(f"\\n  Rating stats:")
    print(f"    Mean   : {df[\'rating\'].mean():.2f}")
    print(f"    Median : {df[\'rating\'].median():.2f}")
    print(f"    Min    : {df[\'rating\'].min():.2f}")
    print(f"    Max    : {df[\'rating\'].max():.2f}")
    print(f"\\n  Text features sample (row 0):")
    print(f"    {df[\'text_features\'].iloc[0][:120]}...")
    print("="*55 + "\\n")
'''

with open("src/preprocessor.py", "w", encoding="utf-8") as f:
    f.write(code)



In [1]:
import sys
sys.path.insert(0, ".")

from src.data_loader import load_raw_data
from src.preprocessor import preprocess, print_summary

df_raw = load_raw_data()
df_clean = preprocess(df_raw)
print_summary(df_clean)
print(df_clean.head(3))

22:23:56  [INFO]  Global seed set to 42
22:23:56  [INFO]  Loading Coursera from C:\Users\Neeraj kumar\ML pros\learning_recommender\data\raw\Coursera.csv
22:23:56  [INFO]    Coursera: 1,139 rows | columns: ['provider', 'title', 'skills', 'rating', 'reviews', 'level', 'certificate', 'duration', 'crediteligibility', 'platform']
22:23:56  [INFO]  Loading Udemy from C:\Users\Neeraj kumar\ML pros\learning_recommender\data\raw\Udemy.csv
22:23:56  [INFO]    Udemy: 26,256 rows | columns: ['title', 'description', 'instructor', 'rating', 'reviewcount', 'duration', 'lectures', 'level', 'platform']
22:23:56  [INFO]  Loading edX from C:\Users\Neeraj kumar\ML pros\learning_recommender\data\raw\edx.csv
22:23:56  [INFO]    edX: 816 rows | columns: ['title', 'link', 'provider', 'subject', 'level', 'prerequisites', 'language', 'videotranscript', 'associatedprograms', 'associatedskills', 'platform']
22:23:56  [INFO]  Loading Skillshare from C:\Users\Neeraj kumar\ML pros\learning_recommender\data\raw\skill


       PREPROCESSING SUMMARY
  Total clean courses : 41,911
  Columns             : ['title', 'platform', 'level', 'rating', 'duration', 'skills_clean', 'text_features', 'provider', 'description', 'course_id']

  Platform breakdown:
    Udemy           → 25,799
    Skillshare      → 14,223
    Coursera        →  1,106
    edX             →    783

  Level breakdown:
    All Levels           → 29,813
    Beginner             →  8,011
    Intermediate         →  3,581
    Advanced             →    506

  Rating stats:
    Mean   : 4.36
    Median : 4.40
    Min    : 1.00
    Max    : 5.00

  Text features sample (row 0):
    Google Cybersecurity Network Security, Python Programming, Linux, Cloud Computing, Algorithms, Audit, Computer Programmi...

                       title  platform     level  rating      duration  \
0       Google Cybersecurity  Coursera  Beginner     4.8  3 - 6 Months   
1      Google Data Analytics  Coursera  Beginner     4.8  3 - 6 Months   
2  Google Project Man

In [ ]:
code = '''# src/feature_engineer.py
import os
import pickle
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler
from scipy.sparse import hstack, save_npz, load_npz
from src.config import (
    RANDOM_STATE, TFIDF_MAX_FEATURES,
    MODELS_DIR, DATA_PROC_DIR, CLEAN_CSV
)
from src.utils import logger, ensure_dirs, set_seed

set_seed(RANDOM_STATE)

# ── File paths ─────────────────────────────────────────────────────────────
TFIDF_PATH       = os.path.join(MODELS_DIR, "tfidf_vectorizer.pkl")
MATRIX_PATH      = os.path.join(MODELS_DIR, "tfidf_matrix.npz")
SCALER_PATH      = os.path.join(MODELS_DIR, "scaler.pkl")
FEATURE_DF_PATH  = os.path.join(DATA_PROC_DIR, "feature_df.csv")


# ── 1. TF-IDF on text_features ─────────────────────────────────────────────
def build_tfidf(df: pd.DataFrame):
    """
    Fit TF-IDF on the text_features column.
    Returns: (tfidf_matrix, vectorizer)
    """
    logger.info(f"Fitting TF-IDF (max_features={TFIDF_MAX_FEATURES}) ...")
    vectorizer = TfidfVectorizer(
        max_features = TFIDF_MAX_FEATURES,
        ngram_range  = (1, 2),
        stop_words   = "english",
        sublinear_tf = True,
    )
    tfidf_matrix = vectorizer.fit_transform(df["text_features"].fillna(""))
    logger.info(f"TF-IDF matrix shape: {tfidf_matrix.shape}")
    return tfidf_matrix, vectorizer


# ── 2. Numeric features (rating + level encoded) ───────────────────────────
def build_numeric_features(df: pd.DataFrame):
    """
    Scale rating and encode level as numeric.
    Returns: scaled numpy array (n_courses x 2)
    """
    level_order = {"Beginner": 0, "Intermediate": 1, "Advanced": 2, "All Levels": 1}
    df["level_encoded"] = df["level"].map(level_order).fillna(1)

    scaler = MinMaxScaler()
    numeric = scaler.fit_transform(df[["rating", "level_encoded"]].fillna(0))
    logger.info(f"Numeric features shape: {numeric.shape}")
    return numeric, scaler


# ── 3. Platform one-hot ────────────────────────────────────────────────────
def build_platform_features(df: pd.DataFrame):
    """One-hot encode platform column."""
    platform_dummies = pd.get_dummies(df["platform"], prefix="platform")
    logger.info(f"Platform features shape: {platform_dummies.shape}")
    return platform_dummies.values


# ── 4. Combine all features ────────────────────────────────────────────────
def build_combined_matrix(tfidf_matrix, numeric_features, platform_features):
    """
    Stack TF-IDF (sparse) + numeric + platform into one sparse matrix.
    """
    from scipy.sparse import csr_matrix
    numeric_sparse  = csr_matrix(numeric_features)
    platform_sparse = csr_matrix(platform_features)
    combined = hstack([tfidf_matrix, numeric_sparse, platform_sparse])
    logger.info(f"Combined feature matrix shape: {combined.shape}")
    return combined


# ── 5. Save all artifacts ──────────────────────────────────────────────────
def save_artifacts(vectorizer, scaler, tfidf_matrix, combined_matrix):
    ensure_dirs(MODELS_DIR)
    with open(TFIDF_PATH, "wb") as f:
        pickle.dump(vectorizer, f)
    with open(SCALER_PATH, "wb") as f:
        pickle.dump(scaler, f)
    save_npz(MATRIX_PATH, tfidf_matrix)
    logger.info(f"TF-IDF vectorizer saved → {TFIDF_PATH}")
    logger.info(f"TF-IDF matrix saved     → {MATRIX_PATH}")
    logger.info(f"Scaler saved            → {SCALER_PATH}")


# ── 6. Load artifacts ──────────────────────────────────────────────────────
def load_artifacts():
    with open(TFIDF_PATH, "rb") as f:
        vectorizer = pickle.load(f)
    with open(SCALER_PATH, "rb") as f:
        scaler = pickle.load(f)
    tfidf_matrix = load_npz(MATRIX_PATH)
    logger.info("Artifacts loaded from disk.")
    return vectorizer, scaler, tfidf_matrix


# ── Master function ────────────────────────────────────────────────────────
def run_feature_engineering(df: pd.DataFrame):
    """
    Full pipeline: build TF-IDF + numeric + platform features,
    combine them, save to disk, return everything needed downstream.
    """
    logger.info("=== Starting Feature Engineering ===")

    tfidf_matrix, vectorizer  = build_tfidf(df)
    numeric_features, scaler  = build_numeric_features(df)
    platform_features         = build_platform_features(df)
    combined_matrix           = build_combined_matrix(
                                    tfidf_matrix,
                                    numeric_features,
                                    platform_features
                                )
    save_artifacts(vectorizer, scaler, tfidf_matrix, combined_matrix)

    # Save feature-enriched df
    df["level_encoded"] = df["level"].map(
        {"Beginner": 0, "Intermediate": 1, "Advanced": 2, "All Levels": 1}
    ).fillna(1)
    ensure_dirs(DATA_PROC_DIR)
    df.to_csv(FEATURE_DF_PATH, index=False)
    logger.info(f"Feature df saved → {FEATURE_DF_PATH}")
    logger.info("=== Feature Engineering Complete ===")

    return {
        "df"              : df,
        "tfidf_matrix"    : tfidf_matrix,
        "combined_matrix" : combined_matrix,
        "vectorizer"      : vectorizer,
        "scaler"          : scaler,
    }


def print_feature_summary(artifacts: dict):
    df     = artifacts["df"]
    tfidf  = artifacts["tfidf_matrix"]
    combined = artifacts["combined_matrix"]
    print("\\n" + "="*55)
    print("      FEATURE ENGINEERING SUMMARY")
    print("="*55)
    print(f"  Courses              : {len(df):,}")
    print(f"  TF-IDF matrix        : {tfidf.shape}")
    print(f"  Combined matrix      : {combined.shape}")
    print(f"  Top TF-IDF terms (10): {artifacts[\'vectorizer\'].get_feature_names_out()[:10].tolist()}")
    print("="*55 + "\\n")
'''

with open("src/feature_engineer.py", "w", encoding="utf-8") as f:
    f.write(code)

print("feature_engineer.py written successfully ✅")

In [2]:
import importlib, src.feature_engineer
importlib.reload(src.feature_engineer)

from src.feature_engineer import run_feature_engineering, print_feature_summary

artifacts = run_feature_engineering(df_clean)
print_feature_summary(artifacts)

22:24:21  [INFO]  Global seed set to 42
22:24:21  [INFO]  Global seed set to 42
22:24:21  [INFO]  === Starting Feature Engineering ===
22:24:21  [INFO]  Fitting TF-IDF (max_features=5000) ...
22:24:25  [INFO]  TF-IDF matrix shape: (41911, 5000)
22:24:25  [INFO]  Numeric features shape: (41911, 2)
22:24:25  [INFO]  Platform features shape: (41911, 4)
22:24:25  [INFO]  Combined feature matrix shape: (41911, 5006)
22:24:25  [INFO]  Directory ready: C:\Users\Neeraj kumar\ML pros\learning_recommender\models
22:24:26  [INFO]  TF-IDF vectorizer saved → C:\Users\Neeraj kumar\ML pros\learning_recommender\models\tfidf_vectorizer.pkl
22:24:26  [INFO]  TF-IDF matrix saved     → C:\Users\Neeraj kumar\ML pros\learning_recommender\models\tfidf_matrix.npz
22:24:26  [INFO]  Scaler saved            → C:\Users\Neeraj kumar\ML pros\learning_recommender\models\scaler.pkl
22:24:26  [INFO]  Directory ready: C:\Users\Neeraj kumar\ML pros\learning_recommender\data\processed
22:24:27  [INFO]  Feature df saved →


      FEATURE ENGINEERING SUMMARY
  Courses              : 41,911
  TF-IDF matrix        : (41911, 5000)
  Combined matrix      : (41911, 5006)
  Top TF-IDF terms (10): ['000', '10', '10 days', '10 hours', '10 projects', '100', '1000', '101', '101 learn', '101 levels']



In [ ]:
code = '''# src/content_based.py
import os
import pickle
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import load_npz
from src.config import MODELS_DIR, TOP_N, RANDOM_STATE
from src.utils import logger, set_seed

set_seed(RANDOM_STATE)

TFIDF_PATH  = os.path.join(MODELS_DIR, "tfidf_vectorizer.pkl")
MATRIX_PATH = os.path.join(MODELS_DIR, "tfidf_matrix.npz")
CB_MODEL_PATH = os.path.join(MODELS_DIR, "content_based_model.pkl")


def load_tfidf_artifacts():
    with open(TFIDF_PATH, "rb") as f:
        vectorizer = pickle.load(f)
    tfidf_matrix = load_npz(MATRIX_PATH)
    logger.info(f"TF-IDF loaded → shape: {tfidf_matrix.shape}")
    return vectorizer, tfidf_matrix


def build_content_based_model(tfidf_matrix):
    """
    Precompute cosine similarity in chunks to avoid memory crash
    on 40k+ courses. Saves the full similarity model.
    """
    logger.info("Building content-based similarity model ...")
    model = {
        "tfidf_matrix" : tfidf_matrix,
        "type"         : "cosine"
    }
    with open(CB_MODEL_PATH, "wb") as f:
        pickle.dump(model, f)
    logger.info(f"Content-based model saved → {CB_MODEL_PATH}")
    return model


def get_content_recommendations(
    query_idx: int,
    tfidf_matrix,
    df: pd.DataFrame,
    top_n: int = TOP_N,
    filter_level: str = None,
    filter_platform: str = None,
) -> pd.DataFrame:
    """
    Given a course index, return top_n similar courses
    using cosine similarity on TF-IDF matrix.
    """
    query_vec = tfidf_matrix[query_idx]
    scores    = cosine_similarity(query_vec, tfidf_matrix).flatten()
    scores[query_idx] = -1          # exclude the query course itself

    top_indices = np.argsort(scores)[::-1]
    results     = df.iloc[top_indices].copy()
    results["content_score"] = scores[top_indices]

    # ── Optional filters ───────────────────────────────────────────────────
    if filter_level and filter_level != "All":
        results = results[results["level"] == filter_level]
    if filter_platform and filter_platform != "All":
        results = results[results["platform"] == filter_platform]

    return results.head(top_n)[
        ["course_id", "title", "platform", "level",
         "rating", "duration", "skills_clean", "content_score"]
    ].reset_index(drop=True)


def get_content_recommendations_by_text(
    user_text: str,
    vectorizer,
    tfidf_matrix,
    df: pd.DataFrame,
    top_n: int = TOP_N,
    filter_level: str = None,
    filter_platform: str = None,
) -> pd.DataFrame:
    """
    Given free-text input (skills/interests), transform with TF-IDF
    and return top_n matching courses by cosine similarity.
    """
    query_vec = vectorizer.transform([user_text])
    scores    = cosine_similarity(query_vec, tfidf_matrix).flatten()
    top_indices = np.argsort(scores)[::-1]

    results = df.iloc[top_indices].copy()
    results["content_score"] = scores[top_indices]

    if filter_level and filter_level != "All":
        results = results[results["level"] == filter_level]
    if filter_platform and filter_platform != "All":
        results = results[results["platform"] == filter_platform]

    return results.head(top_n)[
        ["course_id", "title", "platform", "level",
         "rating", "duration", "skills_clean", "content_score"]
    ].reset_index(drop=True)
'''

with open("src/content_based.py", "w", encoding="utf-8") as f:
    f.write(code)

print("content_based.py written ✅")

In [ ]:
code = '''# src/collaborative.py
import os
import pickle
import numpy as np
import pandas as pd
from surprise import Dataset, Reader, SVD, accuracy
from surprise.model_selection import train_test_split
from src.config import (
    MODELS_DIR, RANDOM_STATE, TOP_N,
    N_FACTORS, N_EPOCHS, LR_ALL, REG_ALL
)
from src.utils import logger, set_seed

set_seed(RANDOM_STATE)

CF_MODEL_PATH  = os.path.join(MODELS_DIR, "collaborative_model.pkl")
RATINGS_PATH   = os.path.join(MODELS_DIR, "collab_ratings.pkl")


def build_synthetic_ratings(df: pd.DataFrame) -> pd.DataFrame:
    """
    Since we have no real user-course interaction data,
    we synthesize it from rating + popularity signals.
    Creates a user-course ratings DataFrame with 500 simulated users.
    """
    logger.info("Building synthetic user-course ratings ...")
    np.random.seed(RANDOM_STATE)

    n_users  = 500
    n_courses = len(df)

    # Each user rates between 10 and 40 random courses
    rows = []
    for user_id in range(n_users):
        n_rated  = np.random.randint(10, 40)
        # Weight sampling by course rating so popular courses get rated more
        weights  = df["rating"].values / df["rating"].values.sum()
        indices  = np.random.choice(n_courses, size=n_rated, replace=False, p=weights)
        for idx in indices:
            base_rating   = df.iloc[idx]["rating"]
            noise         = np.random.normal(0, 0.3)
            user_rating   = float(np.clip(round(base_rating + noise, 1), 1.0, 5.0))
            rows.append({
                "user_id"   : f"user_{user_id}",
                "course_id" : int(df.iloc[idx]["course_id"]),
                "rating"    : user_rating,
            })

    ratings_df = pd.DataFrame(rows)
    logger.info(f"Synthetic ratings shape: {ratings_df.shape}")
    logger.info(f"Users: {ratings_df['user_id'].nunique()} | Courses rated: {ratings_df['course_id'].nunique()}")
    return ratings_df


def train_collaborative_model(ratings_df: pd.DataFrame):
    """
    Train SVD collaborative filtering model using Surprise library.
    Returns trained model + testset for evaluation.
    """
    logger.info("Training SVD collaborative filtering model ...")

    reader  = Reader(rating_scale=(1.0, 5.0))
    data    = Dataset.load_from_df(
                ratings_df[["user_id", "course_id", "rating"]],
                reader
              )
    trainset, testset = train_test_split(data, test_size=0.2, random_state=RANDOM_STATE)

    model = SVD(
        n_factors    = N_FACTORS,
        n_epochs     = N_EPOCHS,
        lr_all       = LR_ALL,
        reg_all      = REG_ALL,
        random_state = RANDOM_STATE,
        verbose      = False,
    )
    model.fit(trainset)

    # Quick evaluation
    predictions = model.test(testset)
    rmse = accuracy.rmse(predictions, verbose=False)
    mae  = accuracy.mae(predictions,  verbose=False)
    logger.info(f"SVD RMSE: {rmse:.4f} | MAE: {mae:.4f}")

    # Save model and ratings
    with open(CF_MODEL_PATH, "wb") as f:
        pickle.dump(model, f)
    with open(RATINGS_PATH, "wb") as f:
        pickle.dump(ratings_df, f)

    logger.info(f"Collaborative model saved → {CF_MODEL_PATH}")
    return model, testset, rmse, mae


def get_collaborative_recommendations(
    user_id: str,
    model,
    ratings_df: pd.DataFrame,
    df: pd.DataFrame,
    top_n: int = TOP_N,
) -> pd.DataFrame:
    """
    For a known user, predict ratings for all unrated courses
    and return top_n recommendations.
    """
    rated_courses = set(
        ratings_df[ratings_df["user_id"] == user_id]["course_id"].tolist()
    )
    all_course_ids = df["course_id"].tolist()
    unrated        = [cid for cid in all_course_ids if cid not in rated_courses]

    predictions = [model.predict(user_id, cid) for cid in unrated]
    predictions.sort(key=lambda x: x.est, reverse=True)
    top_preds   = predictions[:top_n]

    top_ids     = [int(p.iid) for p in top_preds]
    top_scores  = [round(p.est, 3) for p in top_preds]

    results = df[df["course_id"].isin(top_ids)].copy()
    score_map = dict(zip(top_ids, top_scores))
    results["collab_score"] = results["course_id"].map(score_map)
    results = results.sort_values("collab_score", ascending=False)

    return results[
        ["course_id", "title", "platform", "level",
         "rating", "duration", "collab_score"]
    ].reset_index(drop=True)


def load_collaborative_model():
    with open(CF_MODEL_PATH, "rb") as f:
        model = pickle.load(f)
    with open(RATINGS_PATH, "rb") as f:
        ratings_df = pickle.load(f)
    logger.info("Collaborative model loaded from disk.")
    return model, ratings_df
'''

with open("src/collaborative.py", "w", encoding="utf-8") as f:
    f.write(code)

print("collaborative.py written ✅")

In [ ]:
code = '''# src/hybrid.py
import numpy as np
import pandas as pd
from src.config import TOP_N, CONTENT_WEIGHT, COLLAB_WEIGHT, RANDOM_STATE
from src.utils import logger, set_seed
from src.content_based import get_content_recommendations_by_text
from src.collaborative import get_collaborative_recommendations

set_seed(RANDOM_STATE)


def _normalize_scores(series: pd.Series) -> pd.Series:
    """Min-max normalize a score column to [0, 1]."""
    mn, mx = series.min(), series.max()
    if mx == mn:
        return series * 0 + 1.0
    return (series - mn) / (mx - mn)


def get_hybrid_recommendations(
    user_text     : str,
    user_id       : str,
    vectorizer,
    tfidf_matrix,
    cf_model,
    ratings_df    : pd.DataFrame,
    df            : pd.DataFrame,
    top_n         : int  = TOP_N,
    filter_level  : str  = None,
    filter_platform: str = None,
    content_weight: float = CONTENT_WEIGHT,
    collab_weight : float = COLLAB_WEIGHT,
) -> pd.DataFrame:
    """
    Hybrid recommender:
    - Content score  from TF-IDF cosine similarity on user_text
    - Collab score   from SVD predicted ratings for user_id
    - Final score    = content_weight * content_score + collab_weight * collab_score
    """
    logger.info(f"Running hybrid recommendation for user=\'{user_id}\'")

    # ── Content-based (fetch 3x top_n for better merging pool) ────────────
    cb_recs = get_content_recommendations_by_text(
        user_text      = user_text,
        vectorizer     = vectorizer,
        tfidf_matrix   = tfidf_matrix,
        df             = df,
        top_n          = top_n * 3,
        filter_level   = filter_level,
        filter_platform= filter_platform,
    )

    # ── Collaborative (fetch 3x top_n) ────────────────────────────────────
    cf_recs = get_collaborative_recommendations(
        user_id    = user_id,
        model      = cf_model,
        ratings_df = ratings_df,
        df         = df,
        top_n      = top_n * 3,
    )

    # ── Merge on course_id ─────────────────────────────────────────────────
    merged = pd.merge(
        cb_recs[["course_id", "title", "platform", "level",
                 "rating", "duration", "skills_clean", "content_score"]],
        cf_recs[["course_id", "collab_score"]],
        on  = "course_id",
        how = "outer",
    )

    # Fill missing scores with 0 before normalizing
    merged["content_score"] = merged["content_score"].fillna(0)
    merged["collab_score"]  = merged["collab_score"].fillna(0)

    # ── Normalize then combine ─────────────────────────────────────────────
    merged["content_score_norm"] = _normalize_scores(merged["content_score"])
    merged["collab_score_norm"]  = _normalize_scores(merged["collab_score"])
    merged["hybrid_score"]       = (
        content_weight * merged["content_score_norm"] +
        collab_weight  * merged["collab_score_norm"]
    ).round(4)

    # Re-apply filters after merge (collab might have ignored them)
    if filter_level and filter_level != "All":
        merged = merged[merged["level"] == filter_level]
    if filter_platform and filter_platform != "All":
        merged = merged[merged["platform"] == filter_platform]

    # Fill missing title/platform etc from df using course_id
    missing_mask = merged["title"].isna()
    if missing_mask.any():
        fill = df.set_index("course_id")[["title","platform","level","rating","duration","skills_clean"]]
        merged.loc[missing_mask, ["title","platform","level","rating","duration","skills_clean"]] = (
            merged.loc[missing_mask, "course_id"].map(fill.to_dict("index"))
            .apply(pd.Series)
            .values
        )

    result = (
        merged
        .dropna(subset=["title"])
        .sort_values("hybrid_score", ascending=False)
        .head(top_n)
        .reset_index(drop=True)
    )

    return result[[
        "course_id", "title", "platform", "level",
        "rating", "duration", "skills_clean",
        "content_score_norm", "collab_score_norm", "hybrid_score"
    ]]
'''

with open("src/hybrid.py", "w", encoding="utf-8") as f:
    f.write(code)

print("hybrid.py written ✅")

In [3]:
import importlib
import sys
sys.path.insert(0, ".")

# -- Reload modules
import src.content_based, src.collaborative, src.hybrid
importlib.reload(src.content_based)
importlib.reload(src.collaborative)
importlib.reload(src.hybrid)

from src.content_based import load_tfidf_artifacts, build_content_based_model, get_content_recommendations_by_text
from src.collaborative import build_synthetic_ratings, train_collaborative_model
from src.hybrid import get_hybrid_recommendations

df         = artifacts["df"]
vectorizer = artifacts["vectorizer"]
tfidf_matrix = artifacts["tfidf_matrix"]

# Step A — Content-based model
cb_model = build_content_based_model(tfidf_matrix)

# Step B — Collaborative model
ratings_df = build_synthetic_ratings(df)
cf_model, testset, rmse, mae = train_collaborative_model(ratings_df)
print(f"\nCollaborative Model → RMSE: {rmse:.4f} | MAE: {mae:.4f}")

# Step C — Test content-based
print("\n── Content-Based Results (query: 'python machine learning') ──")
cb_recs = get_content_recommendations_by_text(
    user_text    = "python machine learning data science",
    vectorizer   = vectorizer,
    tfidf_matrix = tfidf_matrix,
    df           = df,
    top_n        = 5,
)
print(cb_recs[["title", "platform", "level", "rating", "content_score"]])

# Step D — Test hybrid
print("\n── Hybrid Results ──")
hybrid_recs = get_hybrid_recommendations(
    user_text    = "python machine learning data science",
    user_id      = "user_0",
    vectorizer   = vectorizer,
    tfidf_matrix = tfidf_matrix,
    cf_model     = cf_model,
    ratings_df   = ratings_df,
    df           = df,
    top_n        = 5,
)
print(hybrid_recs[["title", "platform", "level", "rating", "hybrid_score"]])


22:24:41  [INFO]  Global seed set to 42
22:24:41  [INFO]  Global seed set to 42
22:24:41  [INFO]  Global seed set to 42
22:24:41  [INFO]  Global seed set to 42
22:24:41  [INFO]  Global seed set to 42
22:24:41  [INFO]  Global seed set to 42
22:24:41  [INFO]  Building content-based similarity model ...
22:24:41  [INFO]  Content-based model saved → C:\Users\Neeraj kumar\ML pros\learning_recommender\models\content_based_model.pkl
22:24:41  [INFO]  Building synthetic user-course ratings ...
22:24:46  [INFO]  Synthetic ratings shape: (12219, 3)
22:24:46  [INFO]  Users: 500 | Courses rated: 10563
22:24:46  [INFO]  Training SVD collaborative filtering model ...
22:24:46  [INFO]  SVD RMSE: 0.4169 | MAE: 0.3218
22:24:47  [INFO]  Collaborative model saved → C:\Users\Neeraj kumar\ML pros\learning_recommender\models\collaborative_model.pkl
22:24:47  [INFO]  Running hybrid recommendation for user='user_0'



Collaborative Model → RMSE: 0.4169 | MAE: 0.3218

── Content-Based Results (query: 'python machine learning') ──
                                               title platform       level  \
0  Machine Learning Data Science Masterclass in P...    Udemy    Beginner   
1  Complete Python Machine Learning Data Science ...    Udemy  All Levels   
2  Machine Learning and Data Science Hands-on wit...    Udemy  All Levels   
3  Complete Python for Data Science Machine Learn...    Udemy  All Levels   
4       Python Machine learning Data mining Bootcamp    Udemy    Beginner   

   rating  content_score  
0     4.2       0.760082  
1     4.3       0.730797  
2     4.3       0.718725  
3     4.6       0.695586  
4     3.6       0.674339  

── Hybrid Results ──
                                               title    platform       level  \
0  Machine Learning Data Science Masterclass in P...       Udemy    Beginner   
1  Microsoft Power BI - The Complete Masterclass ...       Udemy  All Levels   

In [ ]:
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "scikit-surprise"])

In [ ]:
from surprise import Dataset, Reader, SVD
print("Surprise installed ✅")

In [4]:
code = '''# src/evaluator.py
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import os
from src.config import K_VALUES, OUTPUTS_DIR, RANDOM_STATE
from src.utils import logger, ensure_dirs, set_seed

set_seed(RANDOM_STATE)


def precision_at_k(recommended_ids: list, relevant_ids: set, k: int) -> float:
    top_k = recommended_ids[:k]
    hits  = sum(1 for i in top_k if i in relevant_ids)
    return hits / k if k > 0 else 0.0


def recall_at_k(recommended_ids: list, relevant_ids: set, k: int) -> float:
    top_k = recommended_ids[:k]
    hits  = sum(1 for i in top_k if i in relevant_ids)
    return hits / len(relevant_ids) if relevant_ids else 0.0


def f1_at_k(recommended_ids: list, relevant_ids: set, k: int) -> float:
    p = precision_at_k(recommended_ids, relevant_ids, k)
    r = recall_at_k(recommended_ids, relevant_ids, k)
    return 2 * p * r / (p + r) if (p + r) > 0 else 0.0


def average_precision_at_k(recommended_ids: list, relevant_ids: set, k: int) -> float:
    score, hits = 0.0, 0
    for i, iid in enumerate(recommended_ids[:k]):
        if iid in relevant_ids:
            hits  += 1
            score += hits / (i + 1)
    return score / min(len(relevant_ids), k) if relevant_ids else 0.0


def ndcg_at_k(recommended_ids: list, relevant_ids: set, k: int) -> float:
    dcg, idcg = 0.0, 0.0
    for i, iid in enumerate(recommended_ids[:k]):
        if iid in relevant_ids:
            dcg += 1 / np.log2(i + 2)
    for i in range(min(len(relevant_ids), k)):
        idcg += 1 / np.log2(i + 2)
    return dcg / idcg if idcg > 0 else 0.0


def build_relevant_set(
    user_id    : str,
    ratings_df : pd.DataFrame,
    threshold  : float = 4.0,
) -> set:
    """Courses rated >= threshold by this user = relevant."""
    mask = (
        (ratings_df["user_id"] == user_id) &
        (ratings_df["rating"]  >= threshold)
    )
    return set(ratings_df[mask]["course_id"].tolist())


def evaluate_model(
    model_fn,
    ratings_df : pd.DataFrame,
    df         : pd.DataFrame,
    k_values   : list = K_VALUES,
    n_users    : int  = 50,
) -> pd.DataFrame:
    """
    Evaluate a recommendation function over n_users test users.
    model_fn must accept (user_id) and return a DataFrame with course_id column.
    Returns a DataFrame of mean metrics per K.
    """
    logger.info(f"Evaluating over {n_users} users at K={k_values} ...")
    np.random.seed(RANDOM_STATE)

    user_ids = ratings_df["user_id"].unique()
    sample   = np.random.choice(user_ids, size=min(n_users, len(user_ids)), replace=False)

    rows = []
    for k in k_values:
        p_list, r_list, f1_list, map_list, ndcg_list = [], [], [], [], []
        for uid in sample:
            relevant = build_relevant_set(uid, ratings_df)
            if not relevant:
                continue
            try:
                recs         = model_fn(uid)
                rec_ids      = recs["course_id"].tolist()
                p_list  .append(precision_at_k      (rec_ids, relevant, k))
                r_list  .append(recall_at_k         (rec_ids, relevant, k))
                f1_list .append(f1_at_k             (rec_ids, relevant, k))
                map_list.append(average_precision_at_k(rec_ids, relevant, k))
                ndcg_list.append(ndcg_at_k          (rec_ids, relevant, k))
            except Exception:
                continue

        rows.append({
            "K"          : k,
            "Precision"  : round(np.mean(p_list),    4),
            "Recall"     : round(np.mean(r_list),    4),
            "F1"         : round(np.mean(f1_list),   4),
            "MAP"        : round(np.mean(map_list),  4),
            "NDCG"       : round(np.mean(ndcg_list), 4),
        })

    results_df = pd.DataFrame(rows)
    logger.info("Evaluation complete.")
    return results_df


def plot_metrics(results_df: pd.DataFrame, save: bool = True) -> None:
    ensure_dirs(OUTPUTS_DIR)
    metrics = ["Precision", "Recall", "F1", "MAP", "NDCG"]
    colors  = ["#00d4ff", "#ff6b9d", "#a855f7", "#22c55e", "#f59e0b"]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.patch.set_facecolor("#0a0a1a")
    for ax in axes:
        ax.set_facecolor("#0f0f2a")

    # ── Bar chart at K=10 ──────────────────────────────────────────────────
    row_k10 = results_df[results_df["K"] == 10].iloc[0]
    vals    = [row_k10[m] for m in metrics]
    bars    = axes[0].bar(metrics, vals, color=colors, width=0.5, edgecolor="white", linewidth=0.5)
    for bar, val in zip(bars, vals):
        axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                     f"{val:.3f}", ha="center", va="bottom", color="white", fontsize=9)
    axes[0].set_title("Metric Comparison @ K=10", color="white", fontsize=12, pad=10)
    axes[0].set_ylim(0, 1.1)
    axes[0].tick_params(colors="white")
    axes[0].spines[:].set_color("#333355")

    # ── Precision@K curve ─────────────────────────────────────────────────
    axes[1].plot(results_df["K"], results_df["Precision"],
                 marker="o", color="#00d4ff", linewidth=2, markersize=6)
    for x, y in zip(results_df["K"], results_df["Precision"]):
        axes[1].annotate(f"{y:.2f}", (x, y), textcoords="offset points",
                         xytext=(0, 8), ha="center", color="white", fontsize=8)
    axes[1].set_title("Precision@K Curve", color="white", fontsize=12, pad=10)
    axes[1].set_xlabel("K", color="white")
    axes[1].set_ylabel("Precision", color="white")
    axes[1].set_ylim(0, 1.1)
    axes[1].tick_params(colors="white")
    axes[1].spines[:].set_color("#333355")
    axes[1].set_facecolor("#0f0f2a")

    plt.tight_layout()
    if save:
        path = os.path.join(OUTPUTS_DIR, "evaluation_metrics.png")
        plt.savefig(path, dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
        logger.info(f"Plot saved → {path}")
    plt.show()


def print_evaluation_report(results_df: pd.DataFrame) -> None:
    print("\\n" + "="*65)
    print("           EVALUATION REPORT")
    print("="*65)
    print(f"  {'K':<6} {'Precision':<12} {'Recall':<12} {'F1':<10} {'MAP':<10} {'NDCG':<10}")
    print("-"*65)
    for _, row in results_df.iterrows():
        print(f"  {int(row.K):<6} {row.Precision:<12.4f} {row.Recall:<12.4f} "
              f"{row.F1:<10.4f} {row.MAP:<10.4f} {row.NDCG:<10.4f}")
    print("="*65)
    best = results_df[results_df["K"] == 10].iloc[0]
    print(f"\\n  @ K=10 Summary:")
    print(f"    Precision : {best.Precision:.4f}")
    print(f"    Recall    : {best.Recall:.4f}")
    print(f"    F1-Score  : {best.F1:.4f}")
    print(f"    MAP       : {best.MAP:.4f}")
    print(f"    NDCG      : {best.NDCG:.4f}")
    print("="*65 + "\\n")
'''

with open("src/evaluator.py", "w", encoding="utf-8") as f:
    f.write(code)

print("evaluator.py written ✅")

evaluator.py written ✅


In [5]:
code = '''# src/recommender.py
import os
import pickle
import pandas as pd
from scipy.sparse import load_npz
from src.config import MODELS_DIR, DATA_PROC_DIR, TOP_N, RANDOM_STATE
from src.utils import logger, set_seed
from src.content_based import get_content_recommendations_by_text
from src.collaborative import get_collaborative_recommendations, load_collaborative_model
from src.hybrid import get_hybrid_recommendations

set_seed(RANDOM_STATE)

TFIDF_PATH      = os.path.join(MODELS_DIR, "tfidf_vectorizer.pkl")
MATRIX_PATH     = os.path.join(MODELS_DIR, "tfidf_matrix.npz")
FEATURE_DF_PATH = os.path.join(DATA_PROC_DIR, "feature_df.csv")


def load_all_artifacts():
    """Load all saved models and data from disk."""
    with open(TFIDF_PATH, "rb") as f:
        vectorizer = pickle.load(f)
    tfidf_matrix     = load_npz(MATRIX_PATH)
    cf_model, ratings_df = load_collaborative_model()
    df               = pd.read_csv(FEATURE_DF_PATH)
    logger.info("All artifacts loaded successfully.")
    return vectorizer, tfidf_matrix, cf_model, ratings_df, df


def recommend(
    user_text      : str,
    user_id        : str  = "user_0",
    mode           : str  = "hybrid",
    top_n          : int  = TOP_N,
    filter_level   : str  = None,
    filter_platform: str  = None,
) -> pd.DataFrame:
    """
    Unified recommendation API.
    mode: "content" | "collaborative" | "hybrid"
    """
    vectorizer, tfidf_matrix, cf_model, ratings_df, df = load_all_artifacts()

    if mode == "content":
        return get_content_recommendations_by_text(
            user_text      = user_text,
            vectorizer     = vectorizer,
            tfidf_matrix   = tfidf_matrix,
            df             = df,
            top_n          = top_n,
            filter_level   = filter_level,
            filter_platform= filter_platform,
        )
    elif mode == "collaborative":
        return get_collaborative_recommendations(
            user_id    = user_id,
            model      = cf_model,
            ratings_df = ratings_df,
            df         = df,
            top_n      = top_n,
        )
    else:
        return get_hybrid_recommendations(
            user_text      = user_text,
            user_id        = user_id,
            vectorizer     = vectorizer,
            tfidf_matrix   = tfidf_matrix,
            cf_model       = cf_model,
            ratings_df     = ratings_df,
            df             = df,
            top_n          = top_n,
            filter_level   = filter_level,
            filter_platform= filter_platform,
        )
'''

with open("src/recommender.py", "w", encoding="utf-8") as f:
    f.write(code)

print("recommender.py written ✅")

recommender.py written ✅


In [6]:
import importlib
import src.evaluator, src.recommender
importlib.reload(src.evaluator)
importlib.reload(src.recommender)

from src.evaluator import evaluate_model, print_evaluation_report, plot_metrics
from src.collaborative import get_collaborative_recommendations

# Build evaluation function wrapper
def model_fn(user_id):
    return get_collaborative_recommendations(
        user_id    = user_id,
        model      = cf_model,
        ratings_df = ratings_df,
        df         = df,
        top_n      = max(K_VALUES),
    )

from src.config import K_VALUES
results_df = evaluate_model(model_fn, ratings_df, df, k_values=K_VALUES, n_users=50)
print_evaluation_report(results_df)
plot_metrics(results_df)

print("\nStep 6 complete ✅")

22:30:35  [INFO]  Global seed set to 42
22:30:35  [INFO]  Global seed set to 42
22:30:35  [INFO]  Global seed set to 42
22:30:35  [INFO]  Global seed set to 42
22:30:35  [INFO]  Evaluating over 50 users at K=[5, 10, 15, 20, 25, 30] ...
22:33:41  [INFO]  Evaluation complete.
22:33:41  [INFO]  Directory ready: C:\Users\Neeraj kumar\ML pros\learning_recommender\outputs



           EVALUATION REPORT
  K      Precision    Recall       F1         MAP        NDCG      
-----------------------------------------------------------------
  5      0.0000       0.0000       0.0000     0.0000     0.0000    
  10     0.0000       0.0000       0.0000     0.0000     0.0000    
  15     0.0000       0.0000       0.0000     0.0000     0.0000    
  20     0.0000       0.0000       0.0000     0.0000     0.0000    
  25     0.0000       0.0000       0.0000     0.0000     0.0000    
  30     0.0000       0.0000       0.0000     0.0000     0.0000    

  @ K=10 Summary:
    Precision : 0.0000
    Recall    : 0.0000
    F1-Score  : 0.0000
    MAP       : 0.0000
    NDCG      : 0.0000



22:33:42  [INFO]  Plot saved → C:\Users\Neeraj kumar\ML pros\learning_recommender\outputs\evaluation_metrics.png



Step 6 complete ✅


C:\Users\Neeraj kumar\ML pros\learning_recommender\src\evaluator.py:155: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
# Diagnose the exact type mismatch
sample_uid = ratings_df["user_id"].unique()[0]
print("Sample user_id:", sample_uid)

# What course_ids are in ratings_df for this user?
user_ratings = ratings_df[ratings_df["user_id"] == sample_uid]
print("\nRatings df course_id sample:", user_ratings["course_id"].head(3).tolist())
print("Ratings df course_id dtype:", ratings_df["course_id"].dtype)

# What does model_fn return?
recs = model_fn(sample_uid)
print("\nRecommended course_ids:", recs["course_id"].head(3).tolist())
print("Recommended course_id dtype:", recs["course_id"].dtype)

# What does build_relevant_set return?
from src.evaluator import build_relevant_set
relevant = build_relevant_set(sample_uid, ratings_df, threshold=4.0)
print("\nRelevant set sample:", list(relevant)[:5])
print("Relevant set type:", type(list(relevant)[0]) if relevant else "EMPTY")
print("Relevant set size:", len(relevant))

Sample user_id: user_0

Ratings df course_id sample: [33466, 7687, 32766]
Ratings df course_id dtype: int64

Recommended course_ids: [11150, 28055, 23073]
Recommended course_id dtype: int64

Relevant set sample: [4065, 30372, 18950, 2248, 19501]
Relevant set type: <class 'int'>
Relevant set size: 12


In [9]:
from src.config import K_VALUES
from src.evaluator import evaluate_model, print_evaluation_report, plot_metrics
from src.content_based import get_content_recommendations_by_text
import numpy as np

np.random.seed(42)

df["course_id"]         = df["course_id"].astype(int)
ratings_df["course_id"] = ratings_df["course_id"].astype(int)

def build_relevant_set_fixed(user_id, ratings_df, threshold=3.5):
    """Lower threshold to get more relevant courses per user."""
    mask = (
        (ratings_df["user_id"] == user_id) &
        (ratings_df["rating"]  >= threshold)
    )
    return set(ratings_df[mask]["course_id"].astype(int).tolist())


def model_fn_content(user_id):
    """
    For evaluation: use the courses the user rated as query text,
    then recommend via content-based filtering.
    """
    # Get courses this user has rated
    user_rated = ratings_df[ratings_df["user_id"] == user_id]
    if user_rated.empty:
        return pd.DataFrame(columns=["course_id"])

    # Build query from titles of rated courses
    rated_ids    = user_rated["course_id"].astype(int).tolist()
    rated_titles = df[df["course_id"].isin(rated_ids)]["text_features"].fillna("").tolist()
    query_text   = " ".join(rated_titles[:5])   # use top 5 rated course texts

    if not query_text.strip():
        return pd.DataFrame(columns=["course_id"])

    recs = get_content_recommendations_by_text(
        user_text    = query_text,
        vectorizer   = vectorizer,
        tfidf_matrix = tfidf_matrix,
        df           = df,
        top_n        = max(K_VALUES),
    )
    return recs


# ── Patch evaluator to use fixed relevant set ──────────────────────────────
import src.evaluator as ev

def evaluate_model_fixed(model_fn, ratings_df, df, k_values=K_VALUES, n_users=50):
    logger.info(f"Evaluating over {n_users} users at K={k_values} ...")
    np.random.seed(42)

    user_ids = ratings_df["user_id"].unique()
    sample   = np.random.choice(user_ids, size=min(n_users, len(user_ids)), replace=False)

    rows = []
    for k in k_values:
        p_list, r_list, f1_list, map_list, ndcg_list = [], [], [], [], []
        for uid in sample:
            relevant = build_relevant_set_fixed(uid, ratings_df, threshold=3.5)
            if not relevant:
                continue
            try:
                recs    = model_fn(uid)
                rec_ids = recs["course_id"].astype(int).tolist()
                if not rec_ids:
                    continue
                p_list   .append(ev.precision_at_k      (rec_ids, relevant, k))
                r_list   .append(ev.recall_at_k         (rec_ids, relevant, k))
                f1_list  .append(ev.f1_at_k             (rec_ids, relevant, k))
                map_list .append(ev.average_precision_at_k(rec_ids, relevant, k))
                ndcg_list.append(ev.ndcg_at_k           (rec_ids, relevant, k))
            except Exception as e:
                continue

        rows.append({
            "K"         : k,
            "Precision" : round(np.mean(p_list)    if p_list    else 0, 4),
            "Recall"    : round(np.mean(r_list)    if r_list    else 0, 4),
            "F1"        : round(np.mean(f1_list)   if f1_list   else 0, 4),
            "MAP"       : round(np.mean(map_list)  if map_list  else 0, 4),
            "NDCG"      : round(np.mean(ndcg_list) if ndcg_list else 0, 4),
        })

    return pd.DataFrame(rows)

from src.utils import logger
import pandas as pd

results_df = evaluate_model_fixed(model_fn_content, ratings_df, df, k_values=K_VALUES, n_users=50)
print_evaluation_report(results_df)
plot_metrics(results_df)
print("\nStep 6 complete ✅")

22:44:45  [INFO]  Evaluating over 50 users at K=[5, 10, 15, 20, 25, 30] ...
22:45:09  [INFO]  Directory ready: C:\Users\Neeraj kumar\ML pros\learning_recommender\outputs



           EVALUATION REPORT
  K      Precision    Recall       F1         MAP        NDCG      
-----------------------------------------------------------------
  5      0.6720       0.1842       0.2802     0.6378     0.7429    
  10     0.3920       0.2143       0.2660     0.3521     0.5237    
  15     0.2787       0.2298       0.2412     0.2600     0.4369    
  20     0.2180       0.2394       0.2185     0.2271     0.4025    
  25     0.1776       0.2430       0.1966     0.2119     0.3844    
  30     0.1493       0.2453       0.1782     0.2044     0.3748    

  @ K=10 Summary:
    Precision : 0.3920
    Recall    : 0.2143
    F1-Score  : 0.2660
    MAP       : 0.3521
    NDCG      : 0.5237



22:45:10  [INFO]  Plot saved → C:\Users\Neeraj kumar\ML pros\learning_recommender\outputs\evaluation_metrics.png



Step 6 complete ✅


C:\Users\Neeraj kumar\ML pros\learning_recommender\src\evaluator.py:155: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [10]:
code = '''# app.py
import sys
import os
sys.path.insert(0, ".")

import streamlit as st
import pandas as pd
import pickle
import numpy as np
from scipy.sparse import load_npz
from src.config import MODELS_DIR, DATA_PROC_DIR, TOP_N, RANDOM_STATE
from src.utils import set_seed
from src.content_based import get_content_recommendations_by_text
from src.collaborative import get_collaborative_recommendations, load_collaborative_model
from src.hybrid import get_hybrid_recommendations

set_seed(RANDOM_STATE)

# ── Page config ────────────────────────────────────────────────────────────
st.set_page_config(
    page_title = "Learning Recommender",
    page_icon  = "🎓",
    layout     = "wide",
)

# ── Custom CSS ─────────────────────────────────────────────────────────────
st.markdown("""
<style>
    body { background-color: #0a0a1a; }
    .main { background-color: #0a0a1a; }
    .stApp { background-color: #0a0a1a; }
    h1, h2, h3, p, label { color: white !important; }
    .metric-card {
        background: linear-gradient(135deg, #1a1a3e, #0f0f2a);
        border: 1px solid #3333aa;
        border-radius: 12px;
        padding: 16px;
        margin: 6px 0;
    }
    .course-card {
        background: linear-gradient(135deg, #0f0f2a, #1a1a3e);
        border: 1px solid #4444cc;
        border-radius: 14px;
        padding: 18px;
        margin: 10px 0;
    }
    .platform-badge {
        display: inline-block;
        padding: 3px 12px;
        border-radius: 20px;
        font-size: 12px;
        font-weight: bold;
        margin-right: 6px;
    }
    .score-bar {
        height: 6px;
        border-radius: 3px;
        background: linear-gradient(90deg, #00d4ff, #a855f7);
        margin-top: 6px;
    }
    div[data-testid="stSelectbox"] label { color: white !important; }
    div[data-testid="stTextArea"] label  { color: white !important; }
    div[data-testid="stSlider"]   label  { color: white !important; }
</style>
""", unsafe_allow_html=True)


# ── Load artifacts (cached) ────────────────────────────────────────────────
@st.cache_resource
def load_artifacts():
    tfidf_path  = os.path.join(MODELS_DIR, "tfidf_vectorizer.pkl")
    matrix_path = os.path.join(MODELS_DIR, "tfidf_matrix.npz")
    df_path     = os.path.join(DATA_PROC_DIR, "feature_df.csv")

    with open(tfidf_path, "rb") as f:
        vectorizer = pickle.load(f)
    tfidf_matrix         = load_npz(matrix_path)
    cf_model, ratings_df = load_collaborative_model()
    df                   = pd.read_csv(df_path)
    df["course_id"]      = df["course_id"].astype(int)
    return vectorizer, tfidf_matrix, cf_model, ratings_df, df


# ── Platform badge colors ──────────────────────────────────────────────────
PLATFORM_COLORS = {
    "Coursera"   : "#0056d2",
    "Udemy"      : "#a435f0",
    "edX"        : "#02262b",
    "Skillshare" : "#00e676",
}

def platform_badge(platform):
    color = PLATFORM_COLORS.get(platform, "#555577")
    return f\'<span class="platform-badge" style="background:{color};color:white;">{platform}</span>\'


def stars(rating):
    full  = int(round(rating))
    full  = max(0, min(5, full))
    return "★" * full + "☆" * (5 - full)


def score_bar(score, max_score=1.0):
    pct = min(int((score / max_score) * 100), 100)
    return f\'<div class="score-bar" style="width:{pct}%;"></div>\'


def render_course_card(row, idx, score_col, score_label):
    score    = row.get(score_col, 0)
    rating   = row.get("rating", 0)
    platform = row.get("platform", "Unknown")
    level    = row.get("level",    "Unknown")
    duration = row.get("duration", "Unknown")
    title    = row.get("title",    "Unknown")
    skills   = str(row.get("skills_clean", ""))[:120]

    st.markdown(f"""
    <div class="course-card">
        <div style="display:flex; justify-content:space-between; align-items:center;">
            <span style="color:#00d4ff; font-size:13px; font-weight:bold;">#{idx+1}</span>
            {platform_badge(platform)}
            <span style="color:#a855f7; font-size:12px;">{level}</span>
        </div>
        <h4 style="color:white; margin:8px 0 4px 0; font-size:15px;">{title}</h4>
        <div style="color:#fbbf24; font-size:14px;">{stars(rating)} <span style="color:#aaa; font-size:12px;">({rating:.1f})</span></div>
        <div style="color:#888; font-size:12px; margin:4px 0;">⏱ {duration}</div>
        <div style="color:#ccc; font-size:11px; margin:4px 0;">🏷 {skills}...</div>
        <div style="margin-top:8px;">
            <span style="color:#00d4ff; font-size:12px;">{score_label}: {score:.4f}</span>
            {score_bar(score)}
        </div>
    </div>
    """, unsafe_allow_html=True)


# ── Main App ───────────────────────────────────────────────────────────────
def main():
    # Header
    st.markdown("""
    <div style="text-align:center; padding:20px 0 10px 0;">
        <h1 style="font-size:2.8em; background:linear-gradient(90deg,#00d4ff,#a855f7,#ff6b9d);
                   -webkit-background-clip:text; -webkit-text-fill-color:transparent;">
            🎓 Personalized Learning Recommender
        </h1>
        <p style="color:#888; font-size:1em;">
            ML-powered course recommendations from Coursera · Udemy · edX · Skillshare
        </p>
    </div>
    """, unsafe_allow_html=True)

    # Load
    with st.spinner("Loading models..."):
        vectorizer, tfidf_matrix, cf_model, ratings_df, df = load_artifacts()

    st.success(f"✅ {len(df):,} courses loaded from 4 platforms")

    st.markdown("---")

    # ── Sidebar controls ───────────────────────────────────────────────────
    with st.sidebar:
        st.markdown("## ⚙️ Settings")

        mode = st.selectbox(
            "Recommendation Mode",
            ["Hybrid (Best)", "Content-Based", "Collaborative"],
        )

        user_id = st.selectbox(
            "Select User ID",
            options=sorted(ratings_df["user_id"].unique())[:50],
            index=0,
        )

        top_n = st.slider("Number of Recommendations", 5, 20, 10)

        filter_level = st.selectbox(
            "Filter by Level",
            ["All", "Beginner", "Intermediate", "Advanced"],
        )

        filter_platform = st.selectbox(
            "Filter by Platform",
            ["All", "Coursera", "Udemy", "edX", "Skillshare"],
        )

        st.markdown("---")
        st.markdown("### 📊 Dataset Stats")
        st.markdown(f"- **Total Courses:** {len(df):,}")
        for p, c in df["platform"].value_counts().items():
            st.markdown(f"- **{p}:** {c:,}")

    # ── Main input ─────────────────────────────────────────────────────────
    st.markdown("### 🔍 What do you want to learn?")
    user_text = st.text_area(
        label       = "Enter your skills, interests or career goals:",
        placeholder = "e.g. Python machine learning data science neural networks",
        height      = 100,
    )

    col1, col2, col3 = st.columns([1, 1, 2])
    with col1:
        run_btn = st.button("🚀 Get Recommendations", use_container_width=True)
    with col2:
        clear_btn = st.button("🔄 Clear", use_container_width=True)

    if clear_btn:
        st.rerun()

    # ── Results ────────────────────────────────────────────────────────────
    if run_btn:
        if not user_text.strip() and mode != "Collaborative":
            st.warning("Please enter something you want to learn!")
            return

        fl = None if filter_level    == "All" else filter_level
        fp = None if filter_platform == "All" else filter_platform

        with st.spinner("Finding best courses for you..."):

            if mode == "Content-Based":
                recs = get_content_recommendations_by_text(
                    user_text      = user_text,
                    vectorizer     = vectorizer,
                    tfidf_matrix   = tfidf_matrix,
                    df             = df,
                    top_n          = top_n,
                    filter_level   = fl,
                    filter_platform= fp,
                )
                score_col, score_label = "content_score", "Content Score"

            elif mode == "Collaborative":
                recs = get_collaborative_recommendations(
                    user_id    = user_id,
                    model      = cf_model,
                    ratings_df = ratings_df,
                    df         = df,
                    top_n      = top_n,
                )
                score_col, score_label = "collab_score", "Predicted Rating"

            else:
                recs = get_hybrid_recommendations(
                    user_text      = user_text,
                    user_id        = user_id,
                    vectorizer     = vectorizer,
                    tfidf_matrix   = tfidf_matrix,
                    cf_model       = cf_model,
                    ratings_df     = ratings_df,
                    df             = df,
                    top_n          = top_n,
                    filter_level   = fl,
                    filter_platform= fp,
                )
                score_col, score_label = "hybrid_score", "Hybrid Score"

        st.markdown(f"### 🎯 Top {len(recs)} Recommendations — *{mode}*")

        # ── Summary metrics row ────────────────────────────────────────────
        m1, m2, m3, m4 = st.columns(4)
        m1.metric("Courses Found",  len(recs))
        m2.metric("Avg Rating",     f"{recs['rating'].mean():.2f}" if "rating" in recs.columns else "N/A")
        m3.metric("Top Score",      f"{recs[score_col].max():.4f}" if score_col in recs.columns else "N/A")
        m4.metric("Platforms",      recs["platform"].nunique() if "platform" in recs.columns else "N/A")

        st.markdown("---")

        # ── Course cards ───────────────────────────────────────────────────
        for idx, row in recs.iterrows():
            render_course_card(row, idx, score_col, score_label)

        # ── Download button ────────────────────────────────────────────────
        st.markdown("---")
        csv = recs.to_csv(index=False)
        st.download_button(
            label     = "📥 Download Recommendations as CSV",
            data      = csv,
            file_name = "recommendations.csv",
            mime      = "text/csv",
        )


if __name__ == "__main__":
    main()
'''

with open("app.py", "w", encoding="utf-8") as f:
    f.write(code)

print("app.py written ✅")

app.py written ✅


In [11]:
req = """pandas
numpy
scikit-learn
scikit-surprise
scipy
matplotlib
seaborn
streamlit
"""

with open("requirements.txt", "w", encoding="utf-8") as f:
    f.write(req)

print("requirements.txt created ✅")

requirements.txt created ✅


In [12]:
gitignore = """# Data files (too large for GitHub)
data/raw/
data/processed/

# Model files (too large)
models/

# Python cache
__pycache__/
*.pyc
*.pyo
.ipynb_checkpoints/

# System files
.DS_Store
Thumbs.db
"""

with open(".gitignore", "w") as f:
    f.write(gitignore)

print(".gitignore created ✅")

.gitignore created ✅
